# Three-Qubit Bit-Flip Error Correction

**Learning objectives**

- Encode a logical qubit
- Inject and diagnose a single X error
- Correct and verify the logical state

## Environment setup



In [ ]:
# Run once in a fresh Colab/Jupyter environment, then restart the kernel if prompted.
%pip install -q qiskit qiskit-aer matplotlib pylatexenc scipy sympy

In [ ]:
from importlib.metadata import version
for package in ["qiskit", "qiskit-aer", "matplotlib", "pylatexenc", "scipy", "sympy"]:
    print(f"{package:15}: {version(package)}")

## Encode, corrupt and correct

Ancillas 3 and 4 store parity syndromes. Toffoli-controlled corrections restore any one data-qubit X error.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity
def corrected(error_qubit=None,theta=0.9):
    qc=QuantumCircuit(5); qc.ry(theta,0); qc.cx(0,1); qc.cx(0,2)
    if error_qubit is not None: qc.x(error_qubit)
    qc.cx(0,3); qc.cx(1,3); qc.cx(1,4); qc.cx(2,4) # syndromes s01,s12
    qc.x(4); qc.ccx(3,4,0); qc.x(4)                # 10 -> q0
    qc.ccx(3,4,1)                                  # 11 -> q1
    qc.x(3); qc.ccx(3,4,2); qc.x(3)                # 01 -> q2
    qc.cx(1,4); qc.cx(2,4); qc.cx(0,3); qc.cx(1,3) # clear syndromes
    qc.cx(0,2); qc.cx(0,1)                         # decode
    return qc
target_c=QuantumCircuit(1); target_c.ry(0.9,0); target=Statevector.from_instruction(target_c)
for error in [None,0,1,2]:
    qc=corrected(error); out=partial_trace(Statevector.from_instruction(qc),[1,2,3,4])
    print("error",error,"fidelity",state_fidelity(out,target))
display(corrected(1).draw("mpl",fold=-1))

## Limit of the code

The three-qubit repetition code corrects one bit flip, not arbitrary phase errors or multiple flips.

In [ ]:
qc=corrected(None); qc.x(0); qc.x(1)
print("A two-error experiment requires re-running syndrome correction and is not guaranteed to recover.")